In [10]:
import pandas as pd
import numpy as np

# Загружаем данные
df = pd.read_csv('/home/ria/HSE/lab/new_three/data/old_data/passage_association/h3_passages_ambigious.txt', 
                 sep='\t', 
                 names=['strain_id', 'strain_name', 'passage_type'])

# Посмотрим на размер данных
print(f"Всего штаммов: {len(df)}")
print(f"Первые 5 строк:")
print(df.head())

Всего штаммов: 1832
Первые 5 строк:
            strain_id                 strain_name  passage_type
0   STRAIN3094_2015.0           A/Japan/4276/2014  MONKEYKIDNEY
1  STRAIN13073_2015.0           A/Perth/1053/2014     MDCK;SIAT
2  STRAIN11531_2005.5   A/Australia/NHRC0010/2005    UNPASSAGED
3  STRAIN17396_2005.0         A/New_York/390/2005  MONKEYKIDNEY
4   STRAIN9832_2004.0  A/Hong_Kong/CUHK71313/2003           NaN


In [11]:
#  все уникальные типы культивирования

unique_passages = df['passage_type'].unique()
for i, passage in enumerate(unique_passages):
    print(f"{i+1}. {passage}")

print(f"\nВсего уникальных типов: {len(unique_passages)}")

1. MONKEYKIDNEY
2. MDCK;SIAT
3. UNPASSAGED
4. nan
5. SIAT
6. MDCK
7. MDCK;UNKNOWNCELL
8. EGG
9. UNKNOWNCELL
10. MONKEYKIDNEY;MDCK
11. UNKNOWNCELL;SIAT
12. UNKNOWNCELL;MONKEYKIDNEY;MDCK
13. UNKNOWNCELL;MDCK
14. MONKEYKIDNEY;SIAT
15. CK;EGG
16. SIAT;MDCK
17. SIAT;UNKNOWNCELL
18. EGG;MONKEYKIDNEY;MDCK
19. RMIX;MONKEYKIDNEY
20. MDCK;EGG
21. MONKEYKIDNEY;UNKNOWNCELL
22. EGG;MONKEYKIDNEY
23. VERO
24. EGG;MDCK
25. RMIX;MDCK

Всего уникальных типов: 25


In [19]:
def detailed_categorization(passage_str, clinical_lab_category):
    """Детальная категоризация всех типов"""
    
    if clinical_lab_category == 'CLINICAL':
        return 'CLINICAL'
    
    if pd.isna(passage_str):
        return 'nan'
    
    passage_str = str(passage_str).upper()
    
    # Точные совпадения для специфических комбинаций
    category_map = {
        'MDCK': 'MDCK',
        'MONKEYKIDNEY': 'MONKEYKIDNEY', 
        'EGG': 'EGG',
        'UNKNOWNCELL': 'UNKNOWNCELL',
        'VERO': 'VERO',
        'MDCK;UNKNOWNCELL': 'MDCK;UNKNOWNCELL',
        'MONKEYKIDNEY;MDCK': 'MONKEYKIDNEY;MDCK',
        'UNKNOWNCELL;SIAT': 'UNKNOWNCELL;SIAT',
        'UNKNOWNCELL;MONKEYKIDNEY;MDCK': 'UNKNOWNCELL;MONKEYKIDNEY;MDCK',
        'UNKNOWNCELL;MDCK': 'UNKNOWNCELL;MDCK',
        'MONKEYKIDNEY;SIAT': 'MONKEYKIDNEY;SIAT',
        'CK;EGG': 'CK;EGG',
        'SIAT;MDCK': 'SIAT;MDCK',
        'SIAT;UNKNOWNCELL': 'SIAT;UNKNOWNCELL',
        'EGG;MONKEYKIDNEY;MDCK': 'EGG;MONKEYKIDNEY;MDCK',
        'RMIX;MONKEYKIDNEY': 'RMIX;MONKEYKIDNEY',
        'MDCK;EGG': 'MDCK;EGG',
        'MONKEYKIDNEY;UNKNOWNCELL': 'MONKEYKIDNEY;UNKNOWNCELL',
        'EGG;MONKEYKIDNEY': 'EGG;MONKEYKIDNEY',
        'EGG;MDCK': 'EGG;MDCK',
        'RMIX;MDCK': 'RMIX;MDCK'
    }
    
    # Ищем точное совпадение
    if passage_str in category_map:
        return category_map[passage_str]
    else:
        return 'OTHER'  # Для редких случаев, которые мы не учли

# Применяем детальную категоризацию
df['detailed_category'] = df.apply(lambda row: detailed_categorization(
    row['passage_type'], row['clinical_vs_lab']), axis=1)

print("Детальные категории:")
detailed_counts = df['detailed_category'].value_counts()
print(detailed_counts)

print(f"\nКоличество образцов в каждой группе:")
print("=" * 50)
for category, count in detailed_counts.items():
    print(f"{category:30} {count:4d}")

Детальные категории:
detailed_category
CLINICAL                         450
nan                              414
MDCK                             314
UNKNOWNCELL                      303
MONKEYKIDNEY                     113
UNKNOWNCELL;SIAT                  69
MDCK;UNKNOWNCELL                  41
EGG                               35
MONKEYKIDNEY;MDCK                 34
UNKNOWNCELL;MDCK                  25
UNKNOWNCELL;MONKEYKIDNEY;MDCK     16
SIAT;MDCK                          3
SIAT;UNKNOWNCELL                   2
MONKEYKIDNEY;UNKNOWNCELL           2
RMIX;MONKEYKIDNEY                  2
EGG;MONKEYKIDNEY                   2
CK;EGG                             1
MONKEYKIDNEY;SIAT                  1
EGG;MONKEYKIDNEY;MDCK              1
MDCK;EGG                           1
VERO                               1
EGG;MDCK                           1
RMIX;MDCK                          1
Name: count, dtype: int64

Количество образцов в каждой группе:
CLINICAL                        450
nan       

In [14]:
# Посмотрим на распределение только лабораторных образцов по типам
lab_samples = df[df['clinical_vs_lab'] == 'LABORATORY'].copy()

def get_primary_culture_type(passage_str):
    """Определяем основной тип культуры для лабораторных образцов"""
    if pd.isna(passage_str):
        return 'UNKNOWN'
    
    passage_str = str(passage_str).upper()
    
    # Проверяем основные типы (берем первый в списке как основной)
    if passage_str.startswith('EGG') or ';EGG' in passage_str:
        return 'EGG'
    elif passage_str.startswith('MDCK') or passage_str == 'MDCK':
        return 'MDCK'
    elif passage_str.startswith('MONKEYKIDNEY') or passage_str == 'MONKEYKIDNEY':
        return 'MONKEYKIDNEY'
    elif passage_str.startswith('VERO') or passage_str == 'VERO':
        return 'VERO'
    elif passage_str.startswith('RMIX') or passage_str == 'RMIX':
        return 'RMIX'
    elif passage_str.startswith('CK') or passage_str == 'CK':
        return 'CK'
    elif passage_str.startswith('UNKNOWNCELL') or passage_str == 'UNKNOWNCELL':
        return 'UNKNOWNCELL'
    elif 'SIAT' in passage_str and passage_str.startswith('SIAT'):
        return 'SIAT_LAB'
    else:
        return 'MIXED_LAB'

lab_samples['culture_type'] = lab_samples['passage_type'].apply(get_primary_culture_type)

print("Распределение лабораторных образцов по типам культур:")
culture_counts = lab_samples['culture_type'].value_counts()
print(culture_counts)

print(f"\nМинимум для статистики: обычно нужно >50 образцов на группу")
print("Группы с достаточным количеством данных:")
sufficient_groups = culture_counts[culture_counts >= 50]
print(sufficient_groups)

Распределение лабораторных образцов по типам культур:
culture_type
UNKNOWNCELL     413
MDCK            355
MONKEYKIDNEY    150
EGG              41
SIAT_LAB          5
RMIX              3
VERO              1
Name: count, dtype: int64

Минимум для статистики: обычно нужно >50 образцов на группу
Группы с достаточным количеством данных:
culture_type
UNKNOWNCELL     413
MDCK            355
MONKEYKIDNEY    150
Name: count, dtype: int64


In [18]:
def final_categorization_simple(passage_str, clinical_lab_category):
    """Упрощенная категоризация только для сравнения с CLINICAL"""
    
    if clinical_lab_category == 'CLINICAL':
        return 'CLINICAL'
    elif clinical_lab_category == 'UNKNOWN':
        return 'UNKNOWN'
    
    # Для лабораторных определяем тип
    if pd.isna(passage_str):
        return 'UNKNOWN'
    
    passage_str = str(passage_str).upper()
    
    if passage_str.startswith('MDCK') or passage_str == 'MDCK':
        return 'MDCK'
    elif passage_str.startswith('UNKNOWNCELL') or passage_str == 'UNKNOWNCELL':
        return 'UNKNOWNCELL'
    elif passage_str.startswith('MONKEYKIDNEY') or passage_str == 'MONKEYKIDNEY':
        return 'MONKEYKIDNEY'
    else:
        return 'OTHER_LAB'

# Применяем упрощенную категоризацию
df['final_category'] = df.apply(lambda row: final_categorization_simple(
    row['passage_type'], row['clinical_vs_lab']), axis=1)

print("Финальные категории для анализа:")
final_counts = df['final_category'].value_counts()
print(final_counts)

print(f"\nГруппы для сравнения с CLINICAL ({final_counts['CLINICAL']} образцов):")
lab_groups = final_counts[~final_counts.index.isin(['CLINICAL', 'UNKNOWN'])].sort_values(ascending=False)
print(lab_groups)

Финальные категории для анализа:
final_category
CLINICAL        450
UNKNOWN         414
UNKNOWNCELL     413
MDCK            356
MONKEYKIDNEY    150
OTHER_LAB        49
Name: count, dtype: int64

Группы для сравнения с CLINICAL (450 образцов):
final_category
UNKNOWNCELL     413
MDCK            356
MONKEYKIDNEY    150
OTHER_LAB        49
Name: count, dtype: int64


In [ ]:
def final_categorization(passage_str, clinical_lab_category):
    """Категоризация с UNKNOWN как отдельной лабораторной группой"""
    
    if clinical_lab_category == 'CLINICAL':
        return 'CLINICAL'
    elif clinical_lab_category == 'UNKNOWN':
        return 'UNKNOWN_LAB'  # Считаем лабораторными, но отдельной группой
    
    # Для лабораторных определяем конкретный тип
    if pd.isna(passage_str):
        return 'UNKNOWN_LAB'
    
    passage_str = str(passage_str).upper()
    
    if passage_str.startswith('MDCK') or passage_str == 'MDCK':
        return 'MDCK'
    elif passage_str.startswith('UNKNOWNCELL') or passage_str == 'UNKNOWNCELL':
        return 'UNKNOWNCELL'
    elif passage_str.startswith('MONKEYKIDNEY') or passage_str == 'MONKEYKIDNEY':
        return 'MONKEYKIDNEY'
    else:
        return 'OTHER_LAB'

# Применяем новую категоризацию
df['final_category'] = df.apply(lambda row: final_categorization(
    row['passage_type'], row['clinical_vs_lab']), axis=1)

print("Финальные категории для анализа ассоциаций (EpiStat):")
final_counts = df['final_category'].value_counts()
print(final_counts)

print(f"\nПланируемые сравнения:")
print(f"1. CLINICAL (n={final_counts.get('CLINICAL', 0)}) - эталонная группа")
for category in ['MDCK', 'UNKNOWNCELL', 'MONKEYKIDNEY', 'UNKNOWN_LAB']:
    if category in final_counts:
        print(f"2. {category} (n={final_counts[category]}) vs CLINICAL")

# Сохраняем результат
result_df = df[['strain_id', 'strain_name', 'passage_type', 'final_category']].copy()
result_df.to_csv('/mnt/user-data/outputs/categorized_strains_final.tsv', sep='\t', index=False)
print(f"\nКатегоризация сохранена в: categorized_strains_final.tsv")

Финальные категории для анализа:
final_category
CLINICAL        450
UNKNOWN         414
UNKNOWNCELL     413
MDCK            356
MONKEYKIDNEY    150
OTHER_LAB        49
Name: count, dtype: int64

Группы для сравнения с CLINICAL (450 образцов):
final_category
UNKNOWNCELL     413
MDCK            356
MONKEYKIDNEY    150
OTHER_LAB        49
Name: count, dtype: int64


In [21]:
# Создаем общую категорию LABORATORY
def add_laboratory_category(detailed_cat):
    if detailed_cat in ['CLINICAL', 'nan']:
        return detailed_cat
    else:
        return detailed_cat  # Сохраняем детальную категорию

# Создаем также общую категорию LABORATORY для сравнений
df['laboratory_general'] = df['detailed_category'].apply(
    lambda x: 'LABORATORY' if x not in ['CLINICAL', 'nan'] else x
)

print("Общие категории:")
general_counts = df['laboratory_general'].value_counts()
print(general_counts)

# Сохраняем финальный результат
result_df = df[['strain_id', 'strain_name', 'passage_type', 
                'clinical_vs_lab', 'detailed_category', 'laboratory_general']].copy()

result_df.to_csv('/home/ria/HSE/lab/new_three/data/old_data/passage_association/final_categorized_strains.tsv', 
                 sep='\t', index=False)

print(f"\nГруппы для анализа ассоциаций с культивированием (EpiStat):")
print("=" * 50)
print(f"1. CLINICAL vs LABORATORY: {general_counts['CLINICAL']} vs {general_counts['LABORATORY']}")
print("\nДетальные сравнения с CLINICAL:")

lab_groups = detailed_counts[~detailed_counts.index.isin(['CLINICAL', 'nan'])]
for category, count in lab_groups.sort_values(ascending=False).head(10).items():
    print(f"   CLINICAL vs {category}: 450 vs {count}")

print(f"\nРезультат сохранен в: final_categorized_strains.tsv")

Общие категории:
laboratory_general
LABORATORY    968
CLINICAL      450
nan           414
Name: count, dtype: int64

Группы для анализа ассоциаций с культивированием (EpiStat):
1. CLINICAL vs LABORATORY: 450 vs 968

Детальные сравнения с CLINICAL:
   CLINICAL vs MDCK: 450 vs 314
   CLINICAL vs UNKNOWNCELL: 450 vs 303
   CLINICAL vs MONKEYKIDNEY: 450 vs 113
   CLINICAL vs UNKNOWNCELL;SIAT: 450 vs 69
   CLINICAL vs MDCK;UNKNOWNCELL: 450 vs 41
   CLINICAL vs EGG: 450 vs 35
   CLINICAL vs MONKEYKIDNEY;MDCK: 450 vs 34
   CLINICAL vs UNKNOWNCELL;MDCK: 450 vs 25
   CLINICAL vs UNKNOWNCELL;MONKEYKIDNEY;MDCK: 450 vs 16
   CLINICAL vs SIAT;MDCK: 450 vs 3

Результат сохранен в: final_categorized_strains.tsv


In [26]:
# Загрузим готовую таблицу категоризации
categorized = pd.read_csv('/home/ria/HSE/lab/new_three/data/old_data/passage_association/final_categorized_strains.tsv', sep='\t')